In [1]:
import tkinter as tk
from tkinter import scrolledtext, ttk
import pyperclip
import re
import pandas as pd
import sys
from datetime import timedelta, datetime
from threading import Thread
import keyboard
import pygetwindow as gw
import time
import json
from PIL import Image, ImageTk

In [2]:
import getpass # для определения текущего пользователя позже убрать
env = getpass.getuser()

In [3]:
# 0.1 Куда сохранять эксель
xlsx_path = r"C:\Users\lutzb\Desktop\wt_stats\data.xlsx" if env == 'lutzb' else r"D:\py\wt_stats\data.xlsx"

In [33]:
with pd.ExcelFile(xlsx_path, engine='openpyxl') as xls:
    # Пытаемся прочитать лист 'battles'
    if 'battles' in xls.sheet_names:
        df_for_start_page = pd.read_excel(xls, sheet_name='vehicles', engine='openpyxl')
    else:
        print('ошибка')


In [5]:
df_for_start_page.tail(10)

,battle_id,result,vehicle,premium,sl_corrected,rp_corrected,mp,activity_percent,mission_time,kills,kills_air,crits,assists,base_caps,doubler_used,did_died
1595,55787cd0026421d,Поражение,Turm III,1,10371,1173,360,60.0,0.000833,1,0,1,0,0,1.0,1.0
1596,55787cd0026421d,Поражение,Me 262 A-1a/U1,2,0,0,0,0.0,0.001285,0,0,0,0,0,0.0,0.0
1597,55787cd0026421d,Поражение,Wiesel 1A4,0,4435,460,305,58.0,0.000775,1,0,1,0,0,0.0,1.0
1598,58071c2000f23bb,Победа,M4A3 (105)(Франция),0,3109,815,340,80.0,0.001898,0,0,0,0,1,0.0,1.0
1599,58071c2000f23bb,Победа,Crusader Mk.II(Франция),0,1049,437,345,75.0,0.001481,1,0,1,0,0,0.0,1.0
1600,58071c2000f23bb,Победа,VTT DCA,0,1669,1099,338,65.0,0.005671,0,0,0,2,0,1.0,1.0
1601,58071c2000f23bb,Победа,M10 GMC(Франция),0,2642,680,393,82.0,0.001991,1,0,1,1,0,0.0,1.0
1602,58071c2000f23bb,Победа,F6F-5(Франция),0,1230,139,22,46.0,0.000845,0,0,0,0,0,0.0,1.0
1603,5806e55000f10a6,Победа,F6F-5(Франция),0,615,119,109,50.0,0.000718,0,0,1,0,0,0.0,0.0
1604,5806e55000f10a6,Победа,Crusader Mk.II(Франция),0,8462,3103,2923,100.0,0.006493,3,0,6,3,4,0.0,1.0


In [34]:
# Заполняем did_died и doubler_used для корректного расчета
df_for_start_page['did_died'] = df_for_start_page['did_died'].fillna(1)
df_for_start_page['doubler_used'] = df_for_start_page['doubler_used'].fillna(0)

# Добавляем столбец objectives
df_for_start_page['objectives'] = (
    df_for_start_page['kills'] +
    df_for_start_page['kills_air'] +
    df_for_start_page['crits'] +
    df_for_start_page['assists'] +
    df_for_start_page['base_caps']
)
# Добавляем столбец k/d
df_for_start_page['kd'] = (
    sum(df_for_start_page['kills'], df_for_start_page['kills_air']) / sum(df_for_start_page['did_died'], df_for_start_page['doubler_used'])
)


In [35]:
# Новый датафрейм через groupby по имени машинки
df_for_start_page = df_for_start_page.groupby('vehicle', as_index=False).agg(
    avg_sl = ('sl_corrected', 'mean'),
    avg_rp = ('rp_corrected', 'mean'),
    avg_mp = ('mp', 'mean'),
    battle_count = ('battle_id', 'count'),
    objectives = ('objectives', 'mean'),
    avg_kd = ('kd', 'mean'),
    total_did_died=('did_died', 'sum'),
    total_doubler_used=('doubler_used', 'sum')
)

# Добавить выживаемость
df_for_start_page['survivability'] = (
    (df_for_start_page['battle_count'] - df_for_start_page['total_did_died']) / df_for_start_page['battle_count']
)

df_for_start_page

,vehicle,avg_sl,avg_rp,avg_mp,battle_count,objectives,avg_kd,total_did_died,total_doubler_used,survivability
0,2С25,4349.105263,790.473684,248.578947,19,1.631579,0.975244,19.0,0.0,0.000000
1,2С3М,2021.888889,408.333333,134.666667,9,0.444444,0.975244,9.0,0.0,0.000000
2,AD-4,4185.000000,526.875000,355.625000,8,2.500000,0.975570,8.0,0.0,0.000000
3,AMC.35 (ACG.1),561.333333,520.333333,506.666667,3,3.000000,0.975244,3.0,0.0,0.000000
4,AMD.35,380.125000,662.750000,427.500000,8,2.375000,0.975244,8.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...
145,Т-62М-1,4417.857143,818.571429,273.571429,7,1.857143,0.975244,7.0,0.0,0.000000
146,Т-72А,4870.894737,865.000000,290.315789,19,1.789474,0.975244,19.0,0.0,0.000000
147,Як-9УТ,5956.272727,664.636364,490.454545,11,3.454545,0.975363,11.0,0.0,0.000000
148,◌С-3604,2692.333333,563.333333,409.666667,3,2.333333,0.976113,1.0,0.0,0.666667


In [36]:
df_for_start_page = df_for_start_page[df_for_start_page['battle_count'] >= 10]

In [37]:
# Получаем топ-3 по avg_sl
top3_sl = df_for_start_page.nlargest(3, 'avg_sl')[['vehicle', 'avg_sl']].values

# Получаем топ-3 по avg_rp
top3_rp = df_for_start_page.nlargest(3, 'avg_rp')[['vehicle', 'avg_rp']].values

# Получаем топ-3 по avg_mp
top3_mp = df_for_start_page.nlargest(3, 'avg_mp')[['vehicle', 'avg_mp']].values

# Топы
# По боям
i = df_for_start_page['battle_count'].idxmax()
name_max_battle_count = df_for_start_page.loc[i, 'vehicle']
value_max_battle_count = df_for_start_page.loc[i, 'battle_count']
# По кд
i = df_for_start_page['avg_kd'].idxmax()
name_max_kd = df_for_start_page.loc[i, 'vehicle']
value_max_kd = df_for_start_page.loc[i, 'avg_kd']
# По objectives
i = df_for_start_page['objectives'].idxmax()
name_max_objectives = df_for_start_page.loc[i, 'vehicle']
value_max_objectives = df_for_start_page.loc[i, 'objectives']
# По survivability
i = df_for_start_page['survivability'].idxmax()
name_max_survivability = df_for_start_page.loc[i, 'vehicle']
value_max_survivability = df_for_start_page.loc[i, 'survivability']

# Выводим в f-строке
print(f"""
Топ 3 по SL:
1. {top3_sl[0][0]} ({round(top3_sl[0][1]):_})
2. {top3_sl[1][0]} ({round(top3_sl[1][1]):_})
3. {top3_sl[2][0]} ({round(top3_sl[2][1]):_})

Топ 3 по RP:
1. {top3_rp[0][0]} ({round(top3_rp[0][1]):_})
2. {top3_rp[1][0]} ({round(top3_rp[1][1]):_})
3. {top3_rp[2][0]} ({round(top3_rp[2][1]):_})

Топ 3 по MP:
1. {top3_mp[0][0]} ({round(top3_mp[0][1]):_})
2. {top3_mp[1][0]} ({round(top3_mp[1][1]):_})
3. {top3_mp[2][0]} ({round(top3_mp[2][1]):_})

Топы по:
Боям - {name_max_battle_count} ({value_max_battle_count} боев)
КД - {name_max_kd} ({value_max_kd})
Полезности - {name_max_objectives} ({round(value_max_objectives)} килов, критов, ассистов, баз)
Выживаемости - {name_max_survivability} ({value_max_survivability})


""".replace("_", " "))


Топ 3 по SL:
1. ИС-6 (29 288)
2. T58 (24 971)
3. Т-44 (ПМ) (18 072)

Топ 3 по RP:
1. ИС-6 (4 523)
2. T58 (4 313)
3. Т-44 (ПМ) (2 781)

Топ 3 по MP:
1. M-51 (1 150)
2. Strv m/41 S-I (938)
3. ИС-6 (821)

Топы по:
Боям - T58 (60 боев)
КД - ZSD63/PG87 (0.9760434417350303)
Полезности - M-51 (9 килов, критов, ассистов, баз)
Выживаемости - Leopard I (0.25)





In [ ]:
# Найти максимумы по всем числовым колонкам
numeric_cols = ['avg_sl', 'avg_rp', 'avg_mp']
max_values = df_for_start_page[numeric_cols].max()
print(max_values)

In [ ]:
# Найти индекс строки с максимальным значением
top_sl_name = df_for_start_page['avg_sl'].idxmax()
top_sl_value = round(df_for_start_page.loc[top_sl_name, 'avg_sl'])

top_rp_name = df_for_start_page['avg_rp'].idxmax()
top_rp_value = round(df_for_start_page.loc[top_rp_name, 'avg_rp'])

top_mp_name = df_for_start_page['avg_mp'].idxmax()
top_mp_value = round(df_for_start_page.loc[top_mp_name, 'avg_mp'])


print(f"""
      По SL - {top_sl_name} ({top_sl_value:_})
      По RP - {top_rp_name} ({top_rp_value:_})
      По MP - {top_mp_name} ({top_mp_value:_})
      """.replace("_", " "))

In [ ]:
# Найти индекс строки с максимальным значением
i = df_for_start_page['battle_count'].idxmax()
max_objective_name = df_for_start_page.loc[i, 'vehicle']
max_objective_value = df_for_start_page.loc[i, 'battle_count']
print(max_objective_name, max_objective_value)

In [ ]:
df_for_start_page.query("""
    battle_count >= 5 & 
    avg_sl > 1000 &
    vehicle.str.contains('ИС')
""", engine='python')

In [ ]:
###########

In [6]:
# Функция получения данных для стартовой страницы
def generate_data_for_start_page(xlsx_path):
    """
    Аналитическая функция для StartPageFrame. Открывает xlsx, считает показатели, отдает dict
    """
    # 1. Открываем эксель
    with pd.ExcelFile(xlsx_path, engine='openpyxl') as xls:
    # Пытаемся прочитать лист 'battles'
        if 'battles' in xls.sheet_names:
            df_for_start_page = pd.read_excel(xls, sheet_name='vehicles', engine='openpyxl')
        else:
            print('ошибка чтения стартового xlsx')
    
    # 2. Обрабатываем, готовим df_for_start_page со средними сгруппированный по vehicles

    battle_count = df_for_start_page['battle_id'].nunique()

    # Добавляем столбец objectives
    df_for_start_page['objectives'] = (
        df_for_start_page['kills'] +
        df_for_start_page['kills_air'] +
        df_for_start_page['crits'] +
        df_for_start_page['assists'] +
        df_for_start_page['base_caps']
    )
    # Добавляем столбец k/d
    df_for_start_page['kd'] = (
        (df_for_start_page['kills'] + df_for_start_page['kills_air'] ) /
        (df_for_start_page['did_died'] + df_for_start_page['doubler_used'])
    )

    # Новый датафрейм через groupby по имени машинки
    df_for_start_page = df_for_start_page.groupby('vehicle', as_index=False).agg(
        avg_sl = ('sl_corrected', 'mean'),
        avg_rp = ('rp_corrected', 'mean'),
        avg_mp = ('mp', 'mean'),
        battle_count = ('battle_id', 'count'),
        objectives = ('objectives', 'mean'),
        avg_kd = ('kd', 'mean'),
        total_did_died=('did_died', 'sum'),
        total_doubler_used=('doubler_used', 'sum')
    )

    # Добавляем столбец выживаемость
    df_for_start_page['survivability'] = (
        (df_for_start_page['battle_count'] - df_for_start_page['total_did_died']) / df_for_start_page['battle_count']
    )

    # Убираем технику, у которой менее 5 боев
    df_for_start_page = df_for_start_page[df_for_start_page['battle_count'] >= 5]

    # 3. Рассчитываем нужные показатели
    # Получаем топ-3 по avg_sl
    top3_sl = df_for_start_page.nlargest(3, 'avg_sl')[['vehicle', 'avg_sl']].values

    # Получаем топ-3 по avg_rp
    top3_rp = df_for_start_page.nlargest(3, 'avg_rp')[['vehicle', 'avg_rp']].values

    # Получаем топ-3 по avg_mp
    top3_mp = df_for_start_page.nlargest(3, 'avg_mp')[['vehicle', 'avg_mp']].values

    # Топы
    # По боям
    i = df_for_start_page['battle_count'].idxmax()
    name_max_battle_count = df_for_start_page.loc[i, 'vehicle']
    value_max_battle_count = df_for_start_page.loc[i, 'battle_count']
    # По кд
    i = df_for_start_page['avg_kd'].idxmax()
    name_max_kd = df_for_start_page.loc[i, 'vehicle']
    value_max_kd = df_for_start_page.loc[i, 'avg_kd']
    # По objectives
    i = df_for_start_page['objectives'].idxmax()
    name_max_objectives = df_for_start_page.loc[i, 'vehicle']
    value_max_objectives = df_for_start_page.loc[i, 'objectives']
    # По survivability
    i = df_for_start_page['survivability'].idxmax()
    name_max_survivability = df_for_start_page.loc[i, 'vehicle']
    value_max_survivability = df_for_start_page.loc[i, 'survivability']

    return {
        'top3_sl_name_1': top3_sl[0][0],
        'top3_sl_value_1': round(top3_sl[0][1]),
        'top3_sl_name_2': top3_sl[1][0],
        'top3_sl_value_2': round(top3_sl[1][1]),
        'top3_sl_name_3': top3_sl[2][0],
        'top3_sl_value_3': round(top3_sl[2][1]),
        'top3_rp_name_1': top3_rp[0][0],
        'top3_rp_value_1': round(top3_rp[0][1]),
        'top3_rp_name_2': top3_rp[1][0],
        'top3_rp_value_2': round(top3_rp[1][1]),
        'top3_rp_name_3': top3_rp[2][0],
        'top3_rp_value_3': round(top3_rp[2][1]),
        'top3_mp_name_1': top3_mp[0][0],
        'top3_mp_value_1': round(top3_mp[0][1]),
        'top3_mp_name_2': top3_mp[1][0],
        'top3_mp_value_2': round(top3_mp[1][1]),
        'top3_mp_name_3': top3_mp[2][0],
        'top3_mp_value_3': round(top3_mp[2][1]),
        'name_max_battle_count': name_max_battle_count,
        'value_max_battle_count': value_max_battle_count,
        'name_max_kd': name_max_kd,
        'value_max_kd': value_max_kd,
        'name_max_objectives': name_max_objectives,
        'value_max_objectives': round(value_max_objectives),
        'name_max_survivability': name_max_survivability,
        'value_max_survivability': value_max_survivability,
        'battle_count': battle_count
    }

In [7]:
data = generate_data_for_start_page(xlsx_path)

In [8]:
class StartPageFrame(tk.Frame):
    """
    Фрейм для отображения статистики по xlsx при запуске программы.
    """
    def __init__(self, parent, data_dict, *args, **kwargs):
        super().__init__(parent, *args, **kwargs)
        self.data_dict = data_dict or {}
        self.widgets = {} # Для хранения ссылок на виджеты (опционально, для динамического обновления)
        self.create_widgets()

    def create_widgets(self):
        """Создает и размещает виджеты на фрейме."""
        # Очищаем фрейм на случай повторного использования
        for widget in self.winfo_children():
            widget.destroy()

        # --- Стили ---
        # Можно настроить шрифты и цвета
        header_font = ("Segoe UI", 14, "bold")
        section_font = ("Segoe UI", 12, "bold")
        value_font = ("Consolas", 10) # Consolas для чисел
        small_font = ("Segoe UI", 10)

        # --- Заголовок ---
        header_label = tk.Label(self, text=f"Топы по предыдущим боям ({self.data_dict['battle_count']})", font=header_font, pady=1)
        header_label.grid(row=0, column=0, columnspan=3, sticky="ew", padx=10)

        # --- Топ 3 по SL, RP, MP ---
        # Заголовки для топ-3
        tk.Label(self, text="🐱 По SL", font=section_font).grid(row=1, column=0, sticky="w", padx=(20, 5), pady=5)
        tk.Label(self, text="💡 По RP", font=section_font).grid(row=1, column=1, sticky="w", padx=5, pady=5)
        tk.Label(self, text="🌐 По MP", font=section_font).grid(row=1, column=2, sticky="w", padx=(5, 20), pady=5)

        # Данные топ-3
        # Используем имена и значения напрямую из data_dict
        for i in range(3):
            row = 2 + i
            # SL
            name_key = f'top3_sl_name_{i+1}'
            value_key = f'top3_sl_value_{i+1}'
            name = self.data_dict.get(name_key, "-")
            value = self.data_dict.get(value_key, "-")
            if isinstance(value, int):
                formatted_value = f"{value:_}".replace("_", " ")
            else:
                formatted_value = str(value)
            text_sl = f"{i+1}. {name} ({formatted_value})"
            tk.Label(self, text=text_sl, font=value_font, anchor="w").grid(row=row, column=0, sticky="w", padx=(15, 5))

            # RP
            name_key = f'top3_rp_name_{i+1}'
            value_key = f'top3_rp_value_{i+1}'
            name = self.data_dict.get(name_key, "-")
            value = self.data_dict.get(value_key, "-")
            if isinstance(value, int):
                formatted_value = f"{value:_}".replace("_", " ")
            else:
                formatted_value = str(value)
            text_rp = f"{i+1}. {name} ({formatted_value})"
            tk.Label(self, text=text_rp, font=value_font, anchor="w").grid(row=row, column=1, sticky="w", padx=5)

            # MP
            name_key = f'top3_mp_name_{i+1}'
            value_key = f'top3_mp_value_{i+1}'
            name = self.data_dict.get(name_key, "-")
            value = self.data_dict.get(value_key, "-")
            if isinstance(value, int):
                formatted_value = f"{value:_}".replace("_", " ")
            else:
                formatted_value = str(value)
            text_mp = f"""{i+1}. {name} ({formatted_value})"""
            tk.Label(self, text=text_mp, font=value_font, anchor="w").grid(row=row, column=2, sticky="w", padx=(5, 15))

        # --- Топ 1 по другим параметрам ---
        next_row = 5 # 2 (заголовки) + 3 (топ-3) 
        # Отступ
        tk.Label(self, text="", font=small_font, pady=5).grid(row=next_row, column=0, columnspan=3)
        next_row += 1

        # Заголовок "ТОП 1 по:"
        tk.Label(self, text="🌟 ТОП 1 по:", font=section_font, pady=5).grid(row=next_row, column=0, columnspan=3, sticky="w", padx=10)
        next_row += 1

        # Список параметров для отображения
        top1_params = [
            ('Боям', 'name_max_battle_count', 'value_max_battle_count'),
            ('КД', 'name_max_kd', 'value_max_kd'),
            ('Полезности', 'name_max_objectives', 'value_max_objectives'),
            ('Выживаемости', 'name_max_survivability', 'value_max_survivability')
        ]

        for i, (label_text, name_key, value_key) in enumerate(top1_params):
            name = self.data_dict.get(name_key, "-")
            val = self.data_dict.get(value_key, "-")
            # Форматируем значение, если это число
            if isinstance(val, (int, float)):
                if label_text == 'Выживаемости':
                     # Для выживаемости показываем проценты или доли
                    formatted_val = f"{val:.2f}" if isinstance(val, float) else str(val)
                else:
                    formatted_val = f"{val:_}".replace("_", " ")
            else:
                formatted_val = str(val)
            display_text = f"{label_text}: {name} ({formatted_val})"
            
            tk.Label(self, text=display_text, font=small_font, anchor="w").grid(row=next_row + i, column=0, columnspan=3, sticky="w", padx=15)

        # --- Конфигурация сетки ---
        # Растягиваем столбцы
        self.grid_columnconfigure(0, weight=1)
        self.grid_columnconfigure(1, weight=1)
        self.grid_columnconfigure(2, weight=1)
        # Отключаем автоматическое растягивание строк для предсказуемости
        # Можно включить, если нужно
        
    def update_data(self, new_data_dict):
        """Обновляет данные и пересоздает виджеты."""
        self.data_dict = new_data_dict
        self.create_widgets()

In [11]:
from tkinter import ttk

In [17]:
root = tk.Tk()
root.title("METANIT.COM")
root.geometry("250x200") 
 
# определяем данные для отображения
people = [("Tom", 38, "tom@email.com"), ("Bob", 42, "bob@email.com"), ("Sam", 28, "sam@email.com")]
 
# определяем столбцы
columns = ("name", "age", "email")
 
tree = ttk.Treeview(columns=columns, show="headings")
tree.pack(fill='both', expand=1)
 
# определяем заголовки
tree.heading("name", text="Имя")
tree.heading("age", text="Возраст")
tree.heading("email", text="Email")
 
# добавляем данные
for person in people:
    tree.insert("", 'end', values=person)
 
root.mainloop()

In [ ]:
root = tk.Tk()
root.title("WT Stats Dashboard - Стартовая страница")
data_dict = generate_data_for_start_page(xlsx_path)
start_page_frame = StartPageFrame(parent=root, data_dict=data_dict)
start_page_frame.pack(fill="both", expand=True)
root.mainloop()

In [ ]:
###########

In [70]:
# Открываем оба листа экселя
with pd.ExcelFile(xlsx_path, engine='openpyxl') as xls:
# Пытаемся прочитать лист 'battles'
    if 'vehicles' in xls.sheet_names:
        df_for_start_page = pd.read_excel(xls, sheet_name='vehicles', engine='openpyxl')
    if 'battles' in xls.sheet_names:
        df_for_start_page_battles_sheet = pd.read_excel(xls, sheet_name='battles', engine='openpyxl')
    else:
        print('ошибка чтения стартового xlsx')

In [7]:
df_for_start_page_battles_sheet.tail(5)

,session_id,vehicles,total_sl,total_frp,total_rp,total_mission_points,result,mission,activity_percent,mission_time,battle_type,max_br,br_country,boosters_sl_percent,boosters_rp_percent,is_prem_veh_used,date
1084,55ddc0a0002e131,M-51,37646,5785,5785,2112,Победа,[Превосходство #1] Второе сражение за Эль-Аламейн,94,0.008600,Tank RB,6.3,Israel,NaN,NaN,0.0,2025-10-10 10:02:04.776
1085,55de6b700032175,"Ki-84 Ko(Китай), M41A3(Китай), Т-34-85 Gai",104417,10497,11015,2670,Победа,[Превосходство] Волоколамск,94,0.007419,Tank RB,5.7,China,15.0,NaN,2.0,2025-10-10 10:19:20.213
1086,55793f200267b7d,"AML-90(Израиль), Hovet, Magach 3 (ERA), Rochev",62801,8133,16248,1596,Победа,[Превосходство] 38-я Параллель,93,0.008148,Tank RB,8.0,Israel,NaN,NaN,1.0,2025-10-10 11:49:25.105
1087,557908e00266c7d,"Gepard, SA 313B Alouette II(Германия), Turm III",71748,7849,11636,2069,Победа,[Превосходство] 38-я Параллель,91,0.006609,Tank RB,8.3,Germany,NaN,NaN,1.0,2025-10-10 12:02:26.297
1088,55787cd0026421d,"DF105, Gepard, Leopard I, M48A2 G A2, Me 262 A...",48928,5459,9598,2834,Поражение,[Превосходство] Заброшенный завод,93,0.007083,Tank RB,8.3,Germany,NaN,NaN,3.0,2025-10-10 13:44:32.007


1089

In [101]:
# Получаем общий винрейт и количество боёв
print(f"""
Винрейт - {round(df_for_start_page_battles_sheet.value_counts('result', normalize=True, dropna=True)['Победа'] * 100, 2)}
Боев - {df_for_start_page_battles_sheet.shape[0]}
""")


Винрейт - 56.38
Боев - 1089



In [74]:
## формируем df для топ по sl, frp, mp
# Приводим винрейт
winrate_by_br = (
    df_for_start_page_battles_sheet
    .groupby('max_br')['result']
    .apply(lambda x: (x == 'Победа').mean())
)

# Формируем group by max_br
df_for_best_br = df_for_start_page_battles_sheet.groupby('max_br', as_index=True).agg(
    avg_sl = ('total_sl', 'mean'),
    avg_frp = ('total_frp', 'mean'),
    avg_mp = ('total_mission_points', 'mean'),
    )

df_for_best_br['winrate'] = winrate_by_br

In [ ]:
df_for_best_br.sort_values(by=["avg_sl", "avg_frp", 'avg_mp'],ascending=[False,False,False])
df_for_best_br.nlargest(3, 'avg_mp')

,avg_sl,avg_frp,avg_mp
max_br,,,
7.7,64492.520833,7871.729167,1545.270833
5.7,52920.682243,5946.560748,1561.121495
5.3,49600.500000,4759.500000,1001.000000
4.7,41962.600000,5033.800000,1807.000000
8.3,41644.504274,5738.837607,1459.230769
8.0,39049.571429,6097.285714,1038.285714
6.3,37679.138462,4871.200000,1705.015385
6.7,35494.403226,4680.177419,1633.080645
3.7,30918.666667,6961.333333,1106.666667


In [75]:
# Рассчитываем Z-оценки для выбора лучшего бр
for col in ["avg_sl", "avg_frp", 'avg_mp']:
    df_for_best_br[col + '_z'] = (df_for_best_br[col] - df_for_best_br[col].mean()) / df_for_best_br[col].std()

df_for_best_br['sum_z'] = df_for_best_br[['avg_sl_z', 'avg_frp_z', 'avg_mp_z']].sum(axis=1)
df_for_best_br = df_for_best_br.sort_values('sum_z', ascending=False)

In [76]:
df_for_best_br.iat[0, 0]

64492.520833333336

In [77]:
df_for_best_br.index[0]

7.7

In [102]:
from colorama import Fore, Style

In [103]:
print(f"""
Лучший БР - {df_for_best_br.index[0]}
SL {round(df_for_best_br.iat[0, 0])}
FRP {round(df_for_best_br.iat[0, 1])}
MP {round(df_for_best_br.iat[0, 2])}
WR {round(df_for_best_br.iat[0, 3] * 100)} %

2 БР - {df_for_best_br.index[1]}
{Fore.BLUE}SL {round(df_for_best_br.iat[1, 0])}{Style.RESET_ALL}
FRP {round(df_for_best_br.iat[1, 1])}
MP {round(df_for_best_br.iat[1, 2])}
WR {round(df_for_best_br.iat[1, 3] * 100)} %

3 БР - {df_for_best_br.index[2]}
SL {round(df_for_best_br.iat[2, 0])}
FRP {round(df_for_best_br.iat[2, 1])}
MP {round(df_for_best_br.iat[2, 2])}
WR {round(df_for_best_br.iat[2, 3] * 100)} %
""")


Лучший БР - 7.7
SL 64493
FRP 7872
MP 1545
WR 48 %

2 БР - 5.7
SL 52921
FRP 5947
MP 1561
WR 50 %

3 БР - 4.7
SL 41963
FRP 5034
MP 1807
WR 40 %



In [49]:
df_for_best_br.iloc[0]['avg_sl']

64492.520833333336